# VLM-as-a-Judge

## নোটবুক পরিচিতি

এই repository-তে কাজ করার জন্য কোনো বাস্তব vision model বা প্রকৃত image pixel
নেই, তাই এই script-টি হাত-নেড়া বর্ণনার বদলে ছোট কিন্তু সত্যিই বাস্তব কিছু
তৈরি করে: একটি "scene" হলো একটি স্পষ্ট, কাঠামোবদ্ধ ground-truth data structure
(color/shape/count + একটি spatial relation) যা একটি ছবির প্রতিনিধিত্ব করে —
একটি ছোট CLEVR-শৈলীর scene graph, কোনো ছবি নয়। প্রতিটি scene থেকে একটি
নির্দিষ্ট caption template একটি সম্পূর্ণ সঠিক caption এবং চারটি লেবেলযুক্ত error
variant তৈরি করে (object hallucination, count error, attribute error, spatial
error)। তারপর দুটি judge এই caption-গুলো গ্রেড করে:

1. **GROUNDED judge**-কে scene graph-টা দেওয়া হয় এবং সে একটি ছোট, সৎ
   regex-ভিত্তিক parser দিয়ে caption-এর প্রতিটি count/relation claim পুনরায়
   বের করে, তারপর প্রতিটি claim-কে প্রকৃত ground truth-এর বিরুদ্ধে যাচাই করে।
2. **UNGROUNDED judge** কেবল caption টেক্সটটিই দেখে — কোনো scene নেই — এবং
   তার কাছে থাকা একমাত্র সংকেতে ফিরে যায়: response length। এটি Phase 08
   Lesson 3-এর verbosity bias, কোনো অতিরিক্ত ত্রুটি হিসেবে যোগ করা নয়, বরং
   আক্ষরিক অর্থে সেটিই যা সারবস্তু যাচাই করতে পারে না এমন একটি judge-এর জন্য
   অবশিষ্ট থাকে।

উভয় judge-কে হাজার হাজার বার random scene/caption pair-এ চালানো হয়, এবং
প্রতিটি error type-এর জন্য REAL পরিমাপকৃত pairwise win rate (judge কি সম্পূর্ণ
সঠিক caption-টিকে প্রতিটি error variant-এর চেয়ে পছন্দ করে?) রিপোর্ট করা হয়,
সাথে একটি কার্যকরী single-scene walkthrough।

## কীভাবে চালাবেন

মূল file-টি হলো `example.py` — `python example.py` দিয়ে চলে। notebook-এ একই
কোড cell-by-cell চালানো হয়; শেষ cell-টি `main()` কল করে।

In [ ]:
import random
import re

random.seed(0)

## 0. Scene graph — একটি ছবির symbolic প্রতিনিধি

COLORS, SHAPES, RELATIONS ধ্রুবক, একটি random scene নির্মাতা, এবং একটি scene-এর
জন্য একমাত্র সম্পূর্ণ সঠিক caption (নির্দিষ্ট template থেকে)।

In [ ]:
COLORS = ["red", "blue", "green", "yellow"]
SHAPES = ["circle", "square", "triangle"]
RELATIONS = ["left of", "right of", "above", "below"]
OPPOSITE_RELATION = {"left of": "right of", "right of": "left of", "above": "below", "below": "above"}


def make_random_scene(rng):
    """একটি scene = দুটি স্বতন্ত্র (color, shape) object, প্রতিটির একটি count,
    সাথে তাদের মধ্যে একটি spatial relation। এটিই ENTIRE ground truth -- একটি
    বাস্তব ছবিকে এই কাজের জন্য encode করতে যা যা লাগত সব, pixel-এ লুকানোর বদলে
    স্পষ্ট ও যাচাইযোগ্য করা।"""
    all_combos = [(c, s) for c in COLORS for s in SHAPES]
    (color_a, shape_a), (color_b, shape_b) = rng.sample(all_combos, 2)
    objects = [
        {"color": color_a, "shape": shape_a, "count": rng.randint(1, 3)},
        {"color": color_b, "shape": shape_b, "count": rng.randint(1, 3)},
    ]
    relation = rng.choice(RELATIONS)
    return {
        "objects": objects,
        "relation": (color_a, shape_a, relation, color_b, shape_b),
    }


def true_caption(scene):
    """একটি scene-এর জন্য একমাত্র সম্পূর্ণ সঠিক caption, নির্দিষ্ট template থেকে
    -- ইচ্ছাকৃতভাবে অনমনীয়, যাতে নিচের grounded judge-এর parser-টি আসল NLP-র
    বদলে একটি সহজ, সৎ regex হতে পারে।"""
    obj_sentences = [f"There are {o['count']} {o['color']} {o['shape']}(s)." for o in scene["objects"]]
    ca, sa, rel, cb, sb = scene["relation"]
    rel_sentence = f"The {ca} {sa} is {rel} the {cb} {sb}."
    return " ".join(obj_sentences + [rel_sentence])

## 1. চারটি লেবেলযুক্ত error injection

প্রতিটি injector একটি সঠিক caption নিয়ে একটি নির্দিষ্ট উপায়ে ভুল করে:
hallucination (যে object-এর অস্তিত্বই scene-এ নেই), count error, attribute
(error), এবং spatial error। count/attribute/spatial error-গুলো caption-এর
দৈর্ঘ্য পরিবর্তন করে না — তাই ungrounded judge-এর কাছে কোনো length cue নেই।

In [ ]:
def inject_hallucination(scene, rng):
    """scene-এ একেবারেই নেই এমন একটি object সংমিশ্রণ সম্পর্কে একটি বাক্য যুক্ত
    করা -- caption দাবি করে যে সেখানে নেই এমন কিছু দেখা যাচ্ছে।"""
    present = {(o["color"], o["shape"]) for o in scene["objects"]}
    absent_combos = [(c, s) for c in COLORS for s in SHAPES if (c, s) not in present]
    color, shape = rng.choice(absent_combos)
    fake_count = rng.randint(1, 3)
    return true_caption(scene) + f" There are {fake_count} {color} {shape}(s)."


def inject_count_error(scene, rng):
    """একটি object-এর বলা count-কে একটি ভুল সংখ্যায় বদলানো -- সঠিক caption-এর
    সমান দৈর্ঘ্য, তাই ungrounded judge-এর কাছে কোনো length cue-ই থাকে না।"""
    caption = true_caption(scene)
    obj = rng.choice(scene["objects"])
    wrong_count = obj["count"]
    while wrong_count == obj["count"]:
        wrong_count = rng.randint(1, 4)
    old = f"There are {obj['count']} {obj['color']} {obj['shape']}(s)."
    new = f"There are {wrong_count} {obj['color']} {obj['shape']}(s)."
    return caption.replace(old, new, 1)


def inject_attribute_error(scene, rng):
    """একটি object-এর color-কে এমন একটিতে বদলানো যা আসলে বিদ্যমান কোনো object-এর
    সাথে মেলে না -- আবারও, সঠিক caption-এর তুলনায় কোনো length পরিবর্তন নেই।"""
    caption = true_caption(scene)
    obj = rng.choice(scene["objects"])
    present_combos = {(o["color"], o["shape"]) for o in scene["objects"]}
    wrong_colors = [c for c in COLORS if (c, obj["shape"]) not in present_combos]
    wrong_color = rng.choice(wrong_colors)
    old = f"There are {obj['count']} {obj['color']} {obj['shape']}(s)."
    new = f"There are {obj['count']} {wrong_color} {obj['shape']}(s)."
    return caption.replace(old, new, 1)


def inject_spatial_error(scene, rng):
    """বলা relation-টিকে তার বিপরীতে উল্টে দেওয়া -- আবারও কোনো length পরিবর্তন নেই।"""
    caption = true_caption(scene)
    ca, sa, rel, cb, sb = scene["relation"]
    old = f"The {ca} {sa} is {rel} the {cb} {sb}."
    new = f"The {ca} {sa} is {OPPOSITE_RELATION[rel]} the {cb} {sb}."
    return caption.replace(old, new, 1)


ERROR_INJECTORS = {
    "hallucination": inject_hallucination,
    "count": inject_count_error,
    "attribute": inject_attribute_error,
    "spatial": inject_spatial_error,
}

## 2. GROUNDED judge — আসল claim extraction + আসল ground-truth check

দুটি regex caption-এর সমস্ত count ও relation claim খুঁজে বের করে, এবং প্রতিটি
scene-এর প্রকৃত ground truth-এর বিরুদ্ধে যাচাই করা হয়। স্কোর হলো claim-গুলোর
মধ্যে সত্য-এর ভগ্নাংশ।

In [ ]:
COUNT_CLAIM_RE = re.compile(r"There are (\d+) (\w+) (\w+)\(s\)\.")
RELATION_CLAIM_RE = re.compile(r"The (\w+) (\w+) is (left of|right of|above|below) the (\w+) (\w+)\.")


def grounded_judge(scene, caption):
    """caption টেক্সট থেকে প্রতিটি count/relation claim বের করে আনে এবং scene-এর
    প্রকৃত ground truth-এর বিরুদ্ধে প্রতিটা যাচাই করে। রিটার্ন করে (সত্য claim-এর
    ভগ্নাংশ, (claim_text, is_true, reason) list)।"""
    true_counts = {(o["color"], o["shape"]): o["count"] for o in scene["objects"]}
    claims = []

    for match in COUNT_CLAIM_RE.finditer(caption):
        count, color, shape = int(match.group(1)), match.group(2), match.group(3)
        key = (color, shape)
        if key not in true_counts:
            claims.append((match.group(0), False, "no such object in the scene"))
        elif true_counts[key] != count:
            claims.append((match.group(0), False, f"actual count is {true_counts[key]}"))
        else:
            claims.append((match.group(0), True, "matches the scene"))

    for match in RELATION_CLAIM_RE.finditer(caption):
        claimed = match.groups()
        if claimed == scene["relation"]:
            claims.append((match.group(0), True, "matches the scene"))
        else:
            claims.append((match.group(0), False, "relation does not match the scene"))

    score = sum(is_true for _, is_true, _ in claims) / len(claims)
    return score, claims

## 3. UNGROUNDED judge — scene-তে কোনো অ্যাক্সেস নেই, কেবল response length

এটিই Lesson 3-এর verbosity bias — এখন একটি ছবি-অন্ধ (image-blind) judge-এর কাছে
একমাত্র উপলব্ধ সংকেত।

In [ ]:
def ungrounded_judge(caption, rng, length_coef=0.05, noise_std=1.0):
    return length_coef * len(caption) + rng.gauss(0.0, noise_std)

## 4. কার্যকরী single-scene walkthrough

একটি scene বেছে নেওয়া হয়, তার সঠিক caption ও প্রতিটি error variant-এর জন্য
grounded judge-এর claim-by-claim ফলাফল দেখানো হয় — কোন claim সত্য, কোনটি মিথ্যা
এবং কেন।

In [ ]:
def worked_example():
    print("=" * 78)
    print("A WORKED EXAMPLE: ONE SCENE, ONE CAPTION PER ERROR TYPE")
    print("=" * 78)
    rng = random.Random(7)
    scene = make_random_scene(rng)

    print("Scene (ground truth):")
    for obj in scene["objects"]:
        print(f"  {obj['count']} x {obj['color']} {obj['shape']}")
    ca, sa, rel, cb, sb = scene["relation"]
    print(f"  relation: the {ca} {sa} is {rel} the {cb} {sb}\n")

    correct = true_caption(scene)
    print(f"Correct caption: {correct!r}")
    score, claims = grounded_judge(scene, correct)
    print(f"  grounded judge score: {score:.2f}  (all claims true: {score == 1.0})\n")

    for error_type, inject in ERROR_INJECTORS.items():
        flawed = inject(scene, rng)
        score, claims = grounded_judge(scene, flawed)
        print(f"[{error_type}] caption: {flawed!r}")
        for claim_text, is_true, reason in claims:
            flag = "OK   " if is_true else "FALSE"
            print(f"    {flag}  {claim_text!r}  ({reason})")
        print(f"    grounded judge score: {score:.2f}\n")


worked_example()

## 5. Aggregate pairwise win rate, অনেক random scene-জুড়ে

প্রতিটি error type-এর জন্য 3,000টি random scene। একটি judge "জেতে" যদি সে সম্পূর্ণ
সঠিক caption-টিকে flawed-টির চেয়ে বেশি স্কোর করে।

In [ ]:
def aggregate_demo():
    print("=" * 78)
    print("AGGREGATE: DOES EACH JUDGE PREFER THE CORRECT CAPTION OVER THE FLAWED ONE?")
    print("=" * 78)
    print("For each error type, 3000 random scenes; a judge 'wins' the trial if it")
    print("scores the fully correct caption higher than the flawed one.\n")

    n_trials = 3000
    rng = random.Random(0)

    header = f"{'error type':16}{'grounded win rate':>20}{'ungrounded win rate':>22}{'caption length delta':>24}"
    print(header)
    print("-" * len(header))

    for error_type, inject in ERROR_INJECTORS.items():
        grounded_wins = 0
        ungrounded_wins = 0
        length_deltas = []

        for _ in range(n_trials):
            scene = make_random_scene(rng)
            correct = true_caption(scene)
            flawed = inject(scene, rng)
            length_deltas.append(len(flawed) - len(correct))

            grounded_correct_score, _ = grounded_judge(scene, correct)
            grounded_flawed_score, _ = grounded_judge(scene, flawed)
            if grounded_correct_score > grounded_flawed_score:
                grounded_wins += 1

            ungrounded_correct_score = ungrounded_judge(correct, rng)
            ungrounded_flawed_score = ungrounded_judge(flawed, rng)
            if ungrounded_correct_score > ungrounded_flawed_score:
                ungrounded_wins += 1

        grounded_rate = grounded_wins / n_trials
        ungrounded_rate = ungrounded_wins / n_trials
        avg_length_delta = sum(length_deltas) / len(length_deltas)
        print(f"{error_type:16}{grounded_rate:>19.1%} {ungrounded_rate:>21.1%} "
              f"{avg_length_delta:>+23.1f}")

    print("\n-> The grounded judge wins essentially every trial for every error type --")
    print("   it is CHECKING real structured facts, so there is nothing for a flawed")
    print("   caption to exploit. The ungrounded judge has no image access at all, so")
    print("   for count/attribute/spatial errors (~0 average length delta) it is at")
    print("   pure CHANCE -- its ~50% win rate reflects zero real signal, not partial")
    print("   competence. For hallucination errors (positive length delta: the flawed")
    print("   caption is LONGER), the ungrounded judge's verbosity bias actively")
    print("   prefers the WORSE, hallucinated caption -- its win rate for the correct")
    print("   one is pushed BELOW 50%, the wrong direction entirely. This is Lesson 3's")
    print("   verbosity bias, deployed here as literally the only signal an image-blind")
    print("   judge has left.")


aggregate_demo()

## সবগুলো demo একসাথে: main()

`main()` দুটি demo একই ক্রমে চালায় (মাঝে একটি ফাঁকা line সহ) — মূল
`example.py`-তে এটি `if __name__ == "__main__":` guard-এর ভেতরে; notebook-এ শেষ
cell হিসেবে `main()` কল করা হয়।

In [ ]:
def main():
    worked_example()
    print()
    aggregate_demo()


main()